# 07 — Constraint Reconstruction, v2 Stress Tests

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** use corrupted observations to reconstruct candidate prime structure through residue, sieve-consistency, density constraints, and stress-test conditions.

Notebook chain:

```text
05 → random controls fail
06 → corrupted observations retain recoverable signal
07 → constraints reconstruct candidate structure
07 v2 → stress-test density reconstruction honestly
08 → local gap reconstruction
```

Core claim:

> Filtering restores precision; reconstruction trades precision for recovery.

Important caution:

> Density-guided completions are candidate hypotheses, not certified primes.

v2 adds:

- biased range corruption
- adversarial mod-6-passing composite noise
- reconstructed gap diagnostics
- expected-gap comparison against \(\log x\)
- stricter interpretation text

## 0. Setup

Artifact structure:

```text
07_constraint_reconstruction/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
07_constraint_reconstruction_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "07_constraint_reconstruction"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Constraint Reconstruction"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Given a corrupted observed set \(S\), estimate a candidate prime set \(\hat{P}_N\).

Stages:

1. raw corrupted observation  
2. residue filter  
3. small-prime sieve filter  
4. density-guided candidate completion

v2 adds harder scenarios so density reconstruction cannot look perfect merely by matching global count.

## 2. Metrics

For an estimated set \(\hat{P}_N\):

\[
recovery = \frac{|\hat{P}_N\cap P_N|}{|P_N|}
\]

\[
precision = \frac{|\hat{P}_N\cap P_N|}{|\hat{P}_N|}
\]

\[
F_1 = \frac{2\cdot precision\cdot recovery}{precision+recovery}
\]

Density drift:

\[
drift_{\mathrm{density}} =
\operatorname{mean}_x \frac{||\hat{P}_N\cap[2,x]|-\pi(x)|}{\pi(x)}
\]

Gap drift:

\[
drift_{\mathrm{gap}} =
\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}
\]

where \(g_i=p_{i+1}-p_i\).

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

SCENARIOS = [
    {"name": "mixed_keep_75_noise_25", "kind": "mixed", "keep_fraction": 0.75, "noise_fraction": 0.25},
    {"name": "mixed_keep_50_noise_50", "kind": "mixed", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "mixed_keep_25_noise_100", "kind": "mixed", "keep_fraction": 0.25, "noise_fraction": 1.00},
    {"name": "biased_low_missing_high_noise", "kind": "biased_range", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_50", "kind": "adversarial_mod6", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_100", "kind": "adversarial_mod6", "keep_fraction": 0.25, "noise_fraction": 1.00},
]

FILTER_DEPTHS = [5, 11, 31, 101, int(math.sqrt(N_MAX))]
DENSITY_MODEL = "x_over_log_x_minus_1"
COMPLETION_BIN_COUNT = 32

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "SCENARIOS": SCENARIOS,
    "FILTER_DEPTHS": FILTER_DEPTHS,
    "DENSITY_MODEL": DENSITY_MODEL,
    "COMPLETION_BIN_COUNT": COMPLETION_BIN_COUNT,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Generate reference primes and corrupted observations

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
prime_set = set(reference_primes.tolist())
universe = np.arange(2, N_MAX + 1)
composites = np.array([n for n in universe if n not in prime_set], dtype=int)
mod6_composites = composites[np.isin(composites % 6, [1, 5])]
filter_primes_all = reference_primes[reference_primes <= int(math.sqrt(N_MAX))]

def make_observation(cfg: dict) -> dict:
    name = cfg["name"]
    kind = cfg["kind"]
    keep_fraction = cfg["keep_fraction"]
    noise_fraction = cfg["noise_fraction"]

    keep_count = int(round(keep_fraction * len(reference_primes)))
    noise_count = int(round(noise_fraction * len(reference_primes)))

    if kind == "biased_range":
        # Harder case: preserve low primes, remove many high primes, then add high-range noise.
        midpoint = N_MAX // 2
        low_primes = reference_primes[reference_primes <= midpoint]
        high_primes = reference_primes[reference_primes > midpoint]

        low_keep_count = min(len(low_primes), int(round(0.90 * len(low_primes))))
        remaining_keep = max(0, keep_count - low_keep_count)
        high_keep_count = min(len(high_primes), remaining_keep)

        kept_low = rng.choice(low_primes, size=low_keep_count, replace=False)
        kept_high = rng.choice(high_primes, size=high_keep_count, replace=False) if high_keep_count else np.array([], dtype=int)

        high_composites = composites[composites > midpoint]
        noise_values = rng.choice(high_composites, size=min(noise_count, len(high_composites)), replace=False)

        values = np.sort(np.unique(np.concatenate([kept_low, kept_high, noise_values])))

    elif kind == "adversarial_mod6":
        # Harder noise: composites that pass the first residue constraint.
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(mod6_composites, size=min(noise_count, len(mod6_composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    else:
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(composites, size=min(noise_count, len(composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    return {
        "name": name,
        "kind": kind,
        "keep_fraction": keep_fraction,
        "noise_fraction": noise_fraction,
        "values": values,
    }

observations = [make_observation(cfg) for cfg in SCENARIOS]

summary = {
    "n_max": int(N_MAX),
    "universe_count": int(len(universe)),
    "prime_count": int(len(reference_primes)),
    "composite_count": int(len(composites)),
    "mod6_composite_count": int(len(mod6_composites)),
    "filter_prime_count": int(len(filter_primes_all)),
}

summary, [(o["name"], o["kind"], len(o["values"])) for o in observations]

## 4. Reconstruction functions

In [ ]:
def residue_filter(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = (values == 2) | (values == 3) | np.isin(values % 6, [1, 5])
    return np.sort(values[keep])

def passes_sieve(values: np.ndarray, q_max: int) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = np.ones(len(values), dtype=bool)
    filter_primes = reference_primes[reference_primes <= q_max]

    for q in filter_primes:
        keep &= ((values == q) | (values % q != 0))

    return keep

def sieve_filter(values: np.ndarray, q_max: int) -> np.ndarray:
    return np.sort(values[passes_sieve(values, q_max)])

def pi_model(x: np.ndarray | float) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    safe = np.maximum(x, 3.0)
    denom = np.log(safe) - 1.0
    denom = np.maximum(denom, 1.0)
    return safe / denom

def density_drift(values: np.ndarray, scales: np.ndarray) -> float:
    values = np.sort(values)
    drifts = []
    for x in scales:
        obs_count = int(np.searchsorted(values, x, side="right"))
        prime_count = int(np.searchsorted(reference_primes, x, side="right"))
        drift = abs(obs_count - prime_count) / prime_count if prime_count else 0.0
        drifts.append(drift)
    return float(np.mean(drifts))

def mod6_score(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=int)
    gt3 = values[values > 3]
    if len(gt3) == 0:
        return float("nan")
    return float(np.mean(np.isin(gt3 % 6, [1, 5])))

def gap_drift(values: np.ndarray) -> float:
    values = np.sort(np.unique(values.astype(int)))
    if len(values) < 3:
        return float("nan")
    gaps = np.diff(values)
    anchors = values[:-1]
    expected = np.maximum(np.log(np.maximum(anchors, 3)), 1.0)
    rel = np.abs(gaps - expected) / expected
    # Trim extreme outliers for a stable finite diagnostic.
    return float(np.mean(np.clip(rel, 0, 20)))

def mean_gap(values: np.ndarray) -> float:
    values = np.sort(np.unique(values.astype(int)))
    return float(np.mean(np.diff(values))) if len(values) >= 2 else float("nan")

def metrics(values: np.ndarray, scenario: str, kind: str, stage: str, scales: np.ndarray) -> dict:
    values = np.sort(np.unique(values.astype(int)))
    value_set = set(values.tolist())
    tp = len(value_set & prime_set)
    fp = len(value_set - prime_set)
    fn = len(prime_set - value_set)

    recovery = tp / len(reference_primes)
    precision = tp / len(values) if len(values) else float("nan")
    f1 = 2 * precision * recovery / (precision + recovery) if (precision + recovery) else float("nan")

    return {
        "scenario": scenario,
        "kind": kind,
        "stage": stage,
        "count": int(len(values)),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "recovery": float(recovery),
        "precision": float(precision),
        "f1": float(f1),
        "density_drift": density_drift(values, scales),
        "mod6_score": mod6_score(values),
        "gap_drift": gap_drift(values),
        "mean_gap": mean_gap(values),
    }

def candidate_pool(q_max: int) -> np.ndarray:
    candidates = universe
    candidates = residue_filter(candidates)
    candidates = sieve_filter(candidates, q_max=q_max)
    return candidates

def density_guided_completion(filtered_values: np.ndarray, q_max: int, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    filtered_values = np.sort(np.unique(filtered_values.astype(int)))
    pool = candidate_pool(q_max=q_max)

    existing = set(filtered_values.tolist())
    available = np.array([v for v in pool if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(filtered_values)))
    if needed_total == 0 or len(available) == 0:
        return filtered_values, pd.DataFrame(columns=["candidate", "bin_left", "bin_right", "score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    completion_rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((filtered_values >= left) & (filtered_values <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)

        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)

        # v2 scoring: centeredness + weak local expected-gap preference.
        centered_score = 1.0 - np.abs(bin_available - center) / scale
        expected_gap = np.maximum(np.log(np.maximum(bin_available, 3)), 1.0)

        # distance to nearest already-filtered value; candidates too close to existing points are penalized.
        if len(filtered_values) > 0:
            positions = np.searchsorted(filtered_values, bin_available)
            left_neighbor = np.where(positions > 0, filtered_values[np.maximum(positions - 1, 0)], -10**12)
            right_neighbor = np.where(positions < len(filtered_values), filtered_values[np.minimum(positions, len(filtered_values)-1)], 10**12)
            nearest_gap = np.minimum(np.abs(bin_available - left_neighbor), np.abs(right_neighbor - bin_available))
            gap_score = np.exp(-np.abs(nearest_gap - expected_gap) / expected_gap)
        else:
            gap_score = np.ones(len(bin_available))

        jitter = rng.normal(0, 1e-6, size=len(bin_available))
        scores = 0.70 * centered_score + 0.30 * gap_score + jitter
        order = np.argsort(scores)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]

        score_lookup = dict(zip(bin_available.tolist(), scores.tolist()))
        for c in chosen:
            completion_rows.append({
                "candidate": int(c),
                "bin_left": int(left),
                "bin_right": int(right),
                "score": float(score_lookup[int(c)]),
            })

    completion_df = pd.DataFrame(completion_rows)
    if len(completion_df) > 0:
        completion_df = completion_df.sort_values("score", ascending=False).head(needed_total)
        selected = completion_df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([filtered_values, selected])))
    return reconstructed, completion_df

scales = np.unique(np.logspace(2, np.log10(N_MAX), 70).astype(int))

print("Functions ready")

## 5. Run staged reconstruction

Stages:

1. raw
2. residue filter
3. full sieve filter
4. density-guided reconstruction after full sieve filter

In [ ]:
stage_metrics = []
candidate_completion_frames = []
stage_sets = {}

q_full = int(math.sqrt(N_MAX))
target_count = len(reference_primes)

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]
    raw = np.sort(obs["values"])

    stage_sets[(scenario, "raw")] = raw
    stage_metrics.append(metrics(raw, scenario, kind, "raw", scales))

    mod6 = residue_filter(raw)
    stage_sets[(scenario, "mod6_filter")] = mod6
    stage_metrics.append(metrics(mod6, scenario, kind, "mod6_filter", scales))

    sieve_full = sieve_filter(mod6, q_max=q_full)
    stage_sets[(scenario, "sieve_filter_full")] = sieve_full
    stage_metrics.append(metrics(sieve_full, scenario, kind, "sieve_filter_full", scales))

    reconstructed, completion_df = density_guided_completion(sieve_full, q_max=q_full, target_count=target_count)
    completion_df["scenario"] = scenario
    completion_df["kind"] = kind
    completion_df["stage"] = "density_reconstruction"
    candidate_completion_frames.append(completion_df)

    stage_sets[(scenario, "density_reconstruction")] = reconstructed
    stage_metrics.append(metrics(reconstructed, scenario, kind, "density_reconstruction", scales))

stage_metrics_df = pd.DataFrame(stage_metrics)
candidate_completions_df = pd.concat(candidate_completion_frames, ignore_index=True) if candidate_completion_frames else pd.DataFrame()

stage_metrics_df.head(16), candidate_completions_df.head()

## 6. Filter-depth sweep

Sweep \(Q_{\mathrm{filter}}\) to show precision/recovery tradeoffs as sieve depth increases.

In [ ]:
sweep_rows = []

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]
    raw = np.sort(obs["values"])
    mod6 = residue_filter(raw)

    for q in FILTER_DEPTHS:
        filtered = sieve_filter(mod6, q_max=q)
        sweep_rows.append({
            **metrics(filtered, scenario, kind, f"sieve_q_{q}", scales),
            "q_filter": int(q),
        })

filter_depth_sweep_df = pd.DataFrame(sweep_rows)

filter_depth_sweep_df.head()

## 7. Confusion counts and gap diagnostics

In [ ]:
confusion_counts_df = stage_metrics_df[
    ["scenario", "kind", "stage", "true_positive", "false_positive", "false_negative"]
].copy()

gap_diagnostics_df = stage_metrics_df[
    ["scenario", "kind", "stage", "gap_drift", "mean_gap", "precision", "recovery", "f1"]
].copy()

measurement = {
    "mean_raw_precision": float(stage_metrics_df[stage_metrics_df["stage"] == "raw"]["precision"].mean()),
    "mean_sieve_precision": float(stage_metrics_df[stage_metrics_df["stage"] == "sieve_filter_full"]["precision"].mean()),
    "mean_reconstruction_recovery": float(stage_metrics_df[stage_metrics_df["stage"] == "density_reconstruction"]["recovery"].mean()),
    "mean_reconstruction_precision": float(stage_metrics_df[stage_metrics_df["stage"] == "density_reconstruction"]["precision"].mean()),
    "mean_reconstruction_f1": float(stage_metrics_df[stage_metrics_df["stage"] == "density_reconstruction"]["f1"].mean()),
    "mean_reconstruction_gap_drift": float(stage_metrics_df[stage_metrics_df["stage"] == "density_reconstruction"]["gap_drift"].mean()),
    "candidate_completion_count": int(len(candidate_completions_df)),
    "scenario_count": int(len(observations)),
}

cgcs = {
    "score": measurement["mean_reconstruction_f1"],
    "definition": "mean F1 score of density-guided reconstruction across stress-test scenarios",
    "interpretation": "Filtering restores precision; reconstruction trades precision for recovery; gap drift checks structural realism.",
}

measurement

## 8. Figure 1 — stage precision and recovery

In [ ]:
stage_order = ["raw", "mod6_filter", "sieve_filter_full", "density_reconstruction"]
scenario_order = [cfg["name"] for cfg in SCENARIOS]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(stage_order))

for scenario in scenario_order:
    sub = stage_metrics_df[stage_metrics_df["scenario"] == scenario].set_index("stage").loc[stage_order]
    ax.plot(x, sub["precision"], marker="o", label=f"{scenario} precision")
    ax.plot(x, sub["recovery"], marker="x", linestyle="--", label=f"{scenario} recovery")

ax.set_xticks(x)
ax.set_xticklabels(stage_order, rotation=20)
ax.set_ylim(0, 1.05)
ax.set_title("Stage precision and recovery")
ax.set_xlabel("stage")
ax.set_ylabel("score")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_stage_precision_recovery.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()
fig1_path

## 9. Figure 2 — F1 by stage

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = stage_metrics_df[stage_metrics_df["scenario"] == scenario].set_index("stage").loc[stage_order]
    ax.plot(stage_order, sub["f1"], marker="o", label=scenario)

ax.set_ylim(0, 1.05)
ax.set_title("F1 by reconstruction stage")
ax.set_xlabel("stage")
ax.set_ylabel("F1")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_f1_by_stage.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()
fig2_path

## 10. Figure 3 — filter-depth sweep

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = filter_depth_sweep_df[filter_depth_sweep_df["scenario"] == scenario].sort_values("q_filter")
    ax.plot(sub["q_filter"], sub["precision"], marker="o", label=f"{scenario} precision")
    ax.plot(sub["q_filter"], sub["recovery"], marker="x", linestyle="--", label=f"{scenario} recovery")

ax.set_xscale("log")
ax.set_ylim(0, 1.05)
ax.set_title("Filter depth sweep")
ax.set_xlabel("Q filter")
ax.set_ylabel("score")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_filter_depth_sweep.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()
fig3_path

## 11. Figure 4 — density drift by stage

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = stage_metrics_df[stage_metrics_df["scenario"] == scenario].set_index("stage").loc[stage_order]
    ax.plot(stage_order, sub["density_drift"], marker="o", label=scenario)

ax.set_title("Density drift by stage")
ax.set_xlabel("stage")
ax.set_ylabel("density drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_density_drift_by_stage.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()
fig4_path

## 12. Figure 5 — confusion counts

In [ ]:
plot_conf = confusion_counts_df.copy()
plot_conf["label"] = plot_conf["scenario"] + "\n" + plot_conf["stage"]

fig, ax = plt.subplots(figsize=(15, 6))
x = np.arange(len(plot_conf))
width = 0.26

ax.bar(x - width, plot_conf["true_positive"], width, label="TP")
ax.bar(x, plot_conf["false_positive"], width, label="FP")
ax.bar(x + width, plot_conf["false_negative"], width, label="FN")

ax.set_xticks(x)
ax.set_xticklabels(plot_conf["label"], rotation=70, ha="right", fontsize=8)
ax.set_title("Confusion counts by stage")
ax.set_xlabel("scenario / stage")
ax.set_ylabel("count")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_confusion_counts.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()
fig5_path

## 13. Figure 6 — candidate completion distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

if len(candidate_completions_df) > 0:
    for scenario in scenario_order:
        sub = candidate_completions_df[candidate_completions_df["scenario"] == scenario]
        if len(sub) > 0:
            ax.hist(sub["candidate"], bins=40, alpha=0.35, label=scenario)
else:
    ax.text(0.5, 0.5, "No completions generated", ha="center", va="center", transform=ax.transAxes)

ax.set_title("Candidate completion distribution")
ax.set_xlabel("candidate value")
ax.set_ylabel("frequency")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_candidate_completion_distribution.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()
fig6_path

## 14. Figure 7 — gap drift by stage

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = stage_metrics_df[stage_metrics_df["scenario"] == scenario].set_index("stage").loc[stage_order]
    ax.plot(stage_order, sub["gap_drift"], marker="o", label=scenario)

ax.set_title("Gap drift by stage")
ax.set_xlabel("stage")
ax.set_ylabel("mean relative gap drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_drift_by_stage.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()
fig7_path

## 15. Figure 8 — reconstructed gap versus expected log x

In [ ]:
# Use strongest adversarial scenario as representative gap diagnostic.
representative = "adversarial_mod6_noise_100"
rep_values = stage_sets[(representative, "density_reconstruction")]
rep_values = np.sort(np.unique(rep_values))
rep_gaps = np.diff(rep_values)
rep_x = rep_values[:-1]
expected_gap = np.log(np.maximum(rep_x, 3))

# Downsample for readability.
sample_idx = np.linspace(0, len(rep_gaps)-1, min(500, len(rep_gaps))).astype(int)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(rep_x[sample_idx], rep_gaps[sample_idx], s=10, alpha=0.45, label="reconstructed gaps")
ax.plot(rep_x[sample_idx], expected_gap[sample_idx], linewidth=2, label="expected log x")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Reconstructed gaps versus expected log x")
ax.set_xlabel("x")
ax.set_ylabel("gap")
ax.legend()
ax.grid(True, alpha=0.3)

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_reconstructed_gap_vs_logx.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()
fig8_path

## 16. Interpretation

1. **Raw observations:** include missing primes and composite noise.

2. **Residue filter:** removes easy invalid residues while preserving most primes.

3. **Sieve filter:** restores high precision by removing composite noise.

4. **Density reconstruction:** adds candidate completions to improve recovery, but can overstate success if only global count is evaluated.

5. **Stress tests:** biased range corruption and adversarial mod-6 noise expose where global density reconstruction remains too coarse.

6. **Gap diagnostics:** reconstructed sets need local gap validation before being treated as structurally realistic.

Core statement:

> Sieve constraints clean noise; density constraints propose missing candidates; gap diagnostics test structural realism.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook tested staged reconstruction from corrupted prime observations, including adversarial stress tests.",
    "",
    "The stages were raw observation, residue filtering, sieve filtering, and density-guided reconstruction.",
    "",
    "## Main findings",
    "",
    "Filtering restores precision.",
    "",
    "Reconstruction trades precision for recovery.",
    "",
    "Density-guided completions are candidate hypotheses, not certified primes.",
    "",
    "v2 stress tests show that global density can repair count-scale drift while still requiring local gap validation.",
    "",
    "## Stress tests",
    "",
    "Biased range corruption removes information unevenly across scale.",
    "",
    "Adversarial mod-6 noise passes the easiest residue screen, forcing sieve-depth and gap diagnostics to do more work.",
    "",
    "## Stage behavior",
    "",
    "Raw observations contain both missing primes and composite noise.",
    "",
    "Residue filtering removes easy invalid residues while preserving most primes.",
    "",
    "Sieve filtering removes composite noise and raises precision strongly.",
    "",
    "Density-guided reconstruction adds candidates to reduce missing-count drift, improving recovery at the cost of possible false positives.",
    "",
    "Gap diagnostics test whether reconstructed candidates match local spacing structure rather than only global density.",
    "",
    "## Summary metrics",
    "",
    f"- mean raw precision = {measurement['mean_raw_precision']:.6f}",
    f"- mean sieve-filter precision = {measurement['mean_sieve_precision']:.6f}",
    f"- mean reconstruction recovery = {measurement['mean_reconstruction_recovery']:.6f}",
    f"- mean reconstruction precision = {measurement['mean_reconstruction_precision']:.6f}",
    f"- mean reconstruction F1 = {measurement['mean_reconstruction_f1']:.6f}",
    f"- mean reconstruction gap drift = {measurement['mean_reconstruction_gap_drift']:.6f}",
    f"- candidate completion count = {measurement['candidate_completion_count']}",
    "",
    "## Core result",
    "",
    "Sieve constraints clean noise; density constraints propose missing candidates; gap diagnostics test structural realism.",
    "",
    "## Caution",
    "",
    "Candidate completions are not proofs of primality. They are constraint-consistent hypotheses for missing structure.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path, fig7_path, fig8_path]
figure_titles = [
    "Stage precision and recovery",
    "F1 by stage",
    "Filter depth sweep",
    "Density drift by stage",
    "Confusion counts",
    "Candidate completion distribution",
    "Gap drift by stage",
    "Reconstructed gap versus expected log x",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 17. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
stage_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_stage_metrics.csv"
filter_depth_sweep_path = DATA_DIR / f"{NOTEBOOK_NUM}_filter_depth_sweep.csv"
candidate_completions_path = DATA_DIR / f"{NOTEBOOK_NUM}_candidate_completions.csv"
confusion_counts_path = DATA_DIR / f"{NOTEBOOK_NUM}_confusion_counts.csv"
gap_diagnostics_path = DATA_DIR / f"{NOTEBOOK_NUM}_gap_diagnostics.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
stage_metrics_df.to_csv(stage_metrics_path, index=False)
filter_depth_sweep_df.to_csv(filter_depth_sweep_path, index=False)
candidate_completions_df.to_csv(candidate_completions_path, index=False)
confusion_counts_df.to_csv(confusion_counts_path, index=False)
gap_diagnostics_df.to_csv(gap_diagnostics_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "stage_metrics": str(stage_metrics_path),
        "filter_depth_sweep": str(filter_depth_sweep_path),
        "candidate_completions": str(candidate_completions_path),
        "confusion_counts": str(confusion_counts_path),
        "gap_diagnostics": str(gap_diagnostics_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 07 follows Notebook 06 by moving from recoverability diagnostics to staged reconstruction.",
    "",
    "v2 adds stress tests so global density reconstruction is evaluated more honestly.",
    "",
    "## Reconstruction stages",
    "",
    "1. raw observation",
    "2. residue filter",
    "3. small-prime sieve filter",
    "4. density-guided reconstruction",
    "",
    "## Stress tests",
    "",
    "1. biased range corruption",
    "2. adversarial mod-6-passing composite noise",
    "3. local gap diagnostics after reconstruction",
    "",
    "## Measurements",
    "",
    "1. recovery",
    "2. precision",
    "3. F1",
    "4. density drift",
    "5. mod 6 score",
    "6. confusion counts",
    "7. gap drift",
    "",
    "## Core claim",
    "",
    "Filtering restores precision; reconstruction trades precision for recovery.",
    "",
    "## Caution",
    "",
    "Candidate completions are hypotheses, not certified primes.",
    "",
    "## Handoff",
    "",
    "Notebook 08 should improve candidate completion using local prime-gap statistics.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook reconstructs candidate prime structure from corrupted observations and stress tests global density reconstruction.",
    "",
    r"Precision and recovery are defined by",
    r"\[",
    r"precision = \frac{|\hat{P}_N\cap P_N|}{|\hat{P}_N|},",
    r"\qquad",
    r"recovery = \frac{|\hat{P}_N\cap P_N|}{|P_N|}.",
    r"\]",
    "",
    r"The $F_1$ score is",
    r"\[",
    r"F_1 = \frac{2\cdot precision\cdot recovery}{precision+recovery}.",
    r"\]",
    "",
    r"Gap drift is approximated by",
    r"\[",
    r"drift_{\mathrm{gap}}=\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item mean raw precision $= {measurement['mean_raw_precision']:.6f}$",
    rf"  \item mean sieve-filter precision $= {measurement['mean_sieve_precision']:.6f}$",
    rf"  \item mean reconstruction recovery $= {measurement['mean_reconstruction_recovery']:.6f}$",
    rf"  \item mean reconstruction precision $= {measurement['mean_reconstruction_precision']:.6f}$",
    rf"  \item mean reconstruction $F_1 = {measurement['mean_reconstruction_f1']:.6f}$",
    rf"  \item mean reconstruction gap drift $= {measurement['mean_reconstruction_gap_drift']:.6f}$",
    r"\end{itemize}",
    "",
    r"Filtering restores precision; reconstruction trades precision for recovery.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Constraint Reconstruction}",
    "",
    r"\subsection*{Prime set}",
    r"\[P_N = \{p : p \le N\}.\]",
    "",
    r"\subsection*{Candidate reconstruction}",
    r"\[\hat{P}_N = F(S_{\mathrm{raw}}).\]",
    "",
    r"\subsection*{Precision and recovery}",
    r"\[precision = \frac{|\hat{P}_N \cap P_N|}{|\hat{P}_N|}.\]",
    r"\[recovery = \frac{|\hat{P}_N \cap P_N|}{|P_N|}.\]",
    "",
    r"\subsection*{$F_1$ score}",
    r"\[F_1 = \frac{2\cdot precision\cdot recovery}{precision+recovery}.\]",
    "",
    r"\subsection*{Residue filter}",
    r"\[n=2,3 \quad \text{or} \quad n\equiv 1,5\pmod6.\]",
    "",
    r"\subsection*{Density-guided completion}",
    r"\[\pi_{\mathrm{model}}(x)\approx \frac{x}{\log x - 1}.\]",
    "",
    r"\subsection*{Gap diagnostic}",
    r"\[drift_{\mathrm{gap}}=\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}.\]",
    "",
    r"Candidate completions are hypotheses, not proofs of primality.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, stage_metrics_path, filter_depth_sweep_path, candidate_completions_path, confusion_counts_path, gap_diagnostics_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 18. Export zip

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 19. Next notebook handoff

Next notebook:

```text
08_local_gap_reconstruction.ipynb
```

Purpose:

> improve candidate completion using local prime-gap statistics rather than global density alone.

In [ ]:
next_step = "Notebook 08: local gap reconstruction."
print(next_step)